In [ ]:
# Install required libraries
!pip install -q transformers datasets librosa evaluate jiwer

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
import os
import pandas as pd
import torch
import librosa
import re
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS
# ==========================================
INPUT_PATH = "/kaggle/input/competitions/multilingual-speech-recognition"
TEST_CSV = os.path.join(INPUT_PATH, "test.csv")
DATA_DIR = os.path.join(INPUT_PATH, "competition_data", "test")

# ==========================================
# 2. LOAD MODEL ON CPU (Force Stability)
# ==========================================
print("--- Loading Whisper-Small (CPU Mode) ---")
model_id = "openai/whisper-small" # 'small' is better for multilingual accuracy

# We load them directly to avoid the 'pipeline' metadata errors
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# FORCE CPU
device = torch.device("cpu")
model.to(device)
print("Success: Model loaded on CPU. No more CUDA errors possible.")

# ==========================================
# 3. TRANSCRIPTION LOOP
# ==========================================
test_df = pd.read_csv(TEST_CSV)
predictions = []

print(f"\n--- Transcribing {len(test_df)} files (Estimated time: 5-8 mins) ---")

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(DATA_DIR, row['audio'])
    
    try:
        # STEP A: Manual Audio Loading (Bypasses 'num_frames' error)
        speech, _ = librosa.load(audio_path, sr=16000)
        
        # STEP B: Convert to features
        input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features 
        
        # STEP C: Generate with Beam Search for Accuracy
        # num_beams=5 helps fix that 1.05 WER issue
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features.to(device),
                num_beams=5,
                max_new_tokens=255
            )
        
        # STEP D: Decode and Clean
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # Aggressive Cleaning (Crucial for beating the 0.31 baseline)
        clean_text = transcription.lower().strip()
        clean_text = re.sub(r'[^\w\s]', '', clean_text) # Remove punctuation
        clean_text = " ".join(clean_text.split())       # Remove extra spaces
        
        predictions.append(clean_text)
        
        # Verification Print
        if i % 20 == 0:
            print(f"\nProgress Check [{i}]: {row['audio']} -> {clean_text[:50]}")
            
    except Exception as e:
        print(f"Error at {row['audio']}: {e}")
        predictions.append("") # Keep the row so the CSV matches

# ==========================================
# 4. FINAL SUBMISSION EXPORT
# ==========================================
submission = pd.DataFrame({
    "audio": test_df["audio"],
    "text": predictions
})

# Make sure we didn't lose any rows
if len(submission) == len(test_df):
    submission.to_csv("submission.csv", index=False)
    print("\n--- DONE! submission.csv is ready for the leaderboard ---")
else:
    print(f"ERROR: Row mismatch! Expected {len(test_df)}, got {len(submission)}")

# GPU

In [ ]:
import os
import pandas as pd
import torch
import librosa
import re
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm

# ==========================================
# 1. GPU DIAGNOSTIC & LOADING
# ==========================================
print("--- Step 1: Checking GPU Compatibility ---")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Targeting Device: {device.upper()}")

model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)

# The 'torch_dtype' part is the most important fix for 'no kernel image'
model = WhisperForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True
).to(device)

# ==========================================
# 2. TRANSCRIPTION WITH FP16
# ==========================================
INPUT_PATH = "/kaggle/input/competitions/multilingual-speech-recognition"
DATA_DIR = os.path.join(INPUT_PATH, "competition_data", "test")
test_df = pd.read_csv(os.path.join(INPUT_PATH, "test.csv"))

predictions = []
print(f"--- Transcribing 100 files on {device.upper()} ---")

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(DATA_DIR, row['audio'])
    try:
        # Manual load
        speech, _ = librosa.load(audio_path, sr=16000)
        
        # We must also move inputs to half-precision (fp16) to match the model
        input_features = processor(speech, sampling_rate=16000, return_tensors="pt").input_features
        if device == "cuda":
            input_features = input_features.to(device, dtype=torch.float16)
        
        with torch.no_grad():
            predicted_ids = model.generate(input_features, num_beams=5)
        
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # Clean text
        clean_text = re.sub(r'[^\w\s]', '', transcription.lower().strip())
        predictions.append(" ".join(clean_text.split()))
        
    except Exception as e:
        print(f"Error at {row['audio']}: {e}")
        predictions.append("")

# Save
pd.DataFrame({"audio": test_df["audio"], "text": predictions}).to_csv("submission.csv", index=False)
print("\nSUCCESS: submission.csv created via GPU!")